# Visual Speech Recognition (Lip Reading) with MediaPipe & CNN + BiLSTM

### Updated Project Approach:
- **Face detection:** MediaPipe Face Mesh.
- **Mouth detection:** Lip landmarks to calculate dynamic bounding box for every frame with adaptive margins and temporal fallback.
- **Preprocessing:** Grayscale conversion, aspect-ratio-preserving resize, symmetric zero-padding, and z-score normalisation.
- **Temporal processing:** Preserve the chronological order of mouth movements across frames.
- **Recognition:** Spatiotemporal CNN + Stacked Bidirectional LSTM.
- **Classification:** Train on both complete word and phrase samples.
- **Design Decision:** Treat each complete word or phrase recording as one classification sample using temporal pooling and cross-entropy classification.

In [ ]:
# 1. Dependencies
!pip install opencv-python mediapipe matplotlib tensorflow gdown

In [ ]:
# 2. Imports and Device Setup
import os
import cv2
import glob
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import mediapipe as mp

print('TensorFlow Version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('Available GPUs:', gpus)
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

## 3. Dynamic Lip Extraction (MediaPipe Face Mesh)
Calculates a dynamic bounding box for the mouth in every frame, applying proportional margins and temporal smoothing/fallback if detection briefly drops.

In [ ]:
mp_face_mesh = mp.solutions.face_mesh

# Extract canonical lip landmark indices
LIP_INDICES = sorted(list(set(
    idx for edge in mp_face_mesh.FACEMESH_LIPS for idx in edge
)))

class DynamicLipExtractor:
    def __init__(self, margin_x=0.25, margin_y=0.35):
        self.margin_x = margin_x
        self.margin_y = margin_y
        self.last_bbox = None
        self.face_mesh = mp_face_mesh.FaceMesh(
            static_image_mode=False,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )

    def reset(self):
        self.last_bbox = None

    def extract_lip_roi(self, frame_bgr):
        h, w = frame_bgr.shape[:2]
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        results = self.face_mesh.process(rgb)

        if results.multi_face_landmarks:
            landmarks = results.multi_face_landmarks[0].landmark
            lip_pts = np.array([
                [landmarks[idx].x * w, landmarks[idx].y * h]
                for idx in LIP_INDICES
            ])
            min_x, min_y = np.min(lip_pts, axis=0)
            max_x, max_y = np.max(lip_pts, axis=0)
            pad_x = int((max_x - min_x) * self.margin_x)
            pad_y = int((max_y - min_y) * self.margin_y)
            x1 = max(0, int(min_x - pad_x))
            y1 = max(0, int(min_y - pad_y))
            x2 = min(w, int(max_x + pad_x))
            y2 = min(h, int(max_y + pad_y))

            if self.last_bbox is not None:
                alpha = 0.8
                x1 = int(alpha * x1 + (1 - alpha) * self.last_bbox[0])
                y1 = int(alpha * y1 + (1 - alpha) * self.last_bbox[1])
                x2 = int(alpha * x2 + (1 - alpha) * self.last_bbox[2])
                y2 = int(alpha * y2 + (1 - alpha) * self.last_bbox[3])

            self.last_bbox = (x1, y1, x2, y2)
            return frame_bgr[y1:y2, x1:x2]

        if self.last_bbox is not None:
            x1, y1, x2, y2 = self.last_bbox
            return frame_bgr[y1:y2, x1:x2]

        return frame_bgr[int(h*0.60):int(h*0.88), int(w*0.25):int(w*0.75)]

## 4. Preprocessing: Grayscale, Aspect-Preserving Resize, Padding & Normalisation

In [ ]:
def preprocess_lip_roi(roi, target_size=(64, 96)):
    target_h, target_w = target_size
    if roi is None or roi.size == 0:
        return np.zeros((target_h, target_w, 1), dtype=np.float32)

    # 1. Grayscale conversion
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY) if len(roi.shape) == 3 else roi
    h, w = gray.shape[:2]
    if h == 0 or w == 0:
        return np.zeros((target_h, target_w, 1), dtype=np.float32)

    # 2. Aspect-ratio-preserving resize
    scale = min(target_w / w, target_h / h)
    new_w = max(1, int(w * scale))
    new_h = max(1, int(h * scale))
    resized = cv2.resize(gray, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # 3. Symmetric zero-padding
    canvas = np.zeros((target_h, target_w), dtype=np.float32)
    top = (target_h - new_h) // 2
    left = (target_w - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = resized

    # 4. Normalisation (zero-mean, unit-variance standardisation)
    mean = np.mean(canvas)
    std = np.std(canvas) + 1e-7
    normalized = (canvas - mean) / std

    return np.expand_dims(normalized, axis=-1).astype(np.float32)

In [ ]:
def process_video_sequence(video_path, start_frame=0, end_frame=None, max_frames=75, target_size=(64, 96), extractor=None):
    if extractor is None:
        extractor = DynamicLipExtractor()
    else:
        extractor.reset()

    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            break
        if frame_idx >= start_frame:
            if end_frame is not None and frame_idx > end_frame:
                break
            roi = extractor.extract_lip_roi(frame)
            processed = preprocess_lip_roi(roi, target_size=target_size)
            frames.append(processed)
        frame_idx += 1
    cap.release()

    if len(frames) == 0:
        return np.zeros((max_frames, target_size[0], target_size[1], 1), dtype=np.float32)

    frames = frames[:max_frames]
    if len(frames) < max_frames:
        pad = np.zeros((max_frames - len(frames), target_size[0], target_size[1], 1), dtype=np.float32)
        frames = np.concatenate([np.array(frames, dtype=np.float32), pad], axis=0)
    else:
        frames = np.array(frames, dtype=np.float32)
    return frames

## 5. Visual Verification of Preprocessing Pipeline

In [ ]:
sample_video = './data/s1/bbaf2n.mpg'
if os.path.exists(sample_video):
    test_extractor = DynamicLipExtractor()
    cap = cv2.VideoCapture(sample_video)
    ret, raw_frame = cap.read()
    cap.release()
    if ret:
        roi = test_extractor.extract_lip_roi(raw_frame)
        processed = preprocess_lip_roi(roi)
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 3, 1)
        plt.title('Full Video Frame')
        plt.imshow(cv2.cvtColor(raw_frame, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.subplot(1, 3, 2)
        plt.title('Dynamic Lip ROI')
        plt.imshow(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.subplot(1, 3, 3)
        plt.title('Preprocessed (64x96 Pad & Norm)')
        plt.imshow(processed.squeeze(-1), cmap='gray')
        plt.axis('off')
        plt.tight_layout()
        plt.show()

## 6. Dataset Building: Words & Phrases
Treats each complete word or phrase recording as one classification sample.

In [ ]:
from pipeline.dataset_builder import WordPhraseDatasetBuilder

builder = WordPhraseDatasetBuilder(
    data_dir='./data',
    speaker='s1',
    target_size=(64, 96),
    max_frames=75
)

all_samples = builder.scan_dataset(include_words=True, include_phrases=True)
builder.save_vocabulary('models/vocab_classes.json')

# Train / Validation Split (85% train, 15% validation)
random.seed(42)
random.shuffle(all_samples)
split_idx = int(0.85 * len(all_samples))
train_samples = all_samples[:split_idx]
val_samples = all_samples[split_idx:]

print(f'Train samples: {len(train_samples)} | Validation samples: {len(val_samples)}')
print(f'Total unique classes: {len(builder.classes)}')

train_ds = builder.create_tf_dataset(train_samples, batch_size=8, shuffle=True)
val_ds = builder.create_tf_dataset(val_samples, batch_size=8, shuffle=False)

## 7. Model Architecture: CNN + BiLSTM Classifier

In [ ]:
from pipeline.model import build_cnn_bilstm_classifier

model = build_cnn_bilstm_classifier(
    input_shape=(75, 64, 96, 1),
    num_classes=len(builder.classes),
    cnn_type='3d',
    lstm_units=128,
    dropout_rate=0.4
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

## 8. Training with Early Stopping & Checkpoints

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

callbacks = [
    ModelCheckpoint(
        'models/best_classifier.weights.h5',
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    steps_per_epoch=max(1, len(train_samples) // 8),
    validation_data=val_ds,
    validation_steps=max(1, len(val_samples) // 8),
    epochs=50,
    callbacks=callbacks
)

## 9. Evaluation & Predictions

In [ ]:
# Test prediction on a sample video
test_sample_path = './data/s1/bbaf2n.mpg'
if os.path.exists(test_sample_path):
    test_frames = process_video_sequence(test_sample_path, max_frames=75, target_size=(64, 96))
    input_tensor = np.expand_dims(test_frames, axis=0)
    pred_probs = model.predict(input_tensor)[0]
    top_idx = np.argmax(pred_probs)
    predicted_label = builder.idx_to_class[top_idx]
    confidence = pred_probs[top_idx] * 100.0
    print('Sample:', test_sample_path)
    print(f'Predicted Class: {predicted_label} ({confidence:.2f}% confidence)')